In [ ]:
# Lab type: debug
# Course: AI401 — AI Applications with LLMs
# Lesson: Output Validation Layers: Schema, Semantic, and Behavioural
# Task: Find and fix 3 bugs in a three-layer validation pipeline for a ticket classifier

# Lab: Debugging a Three-Layer Validation Pipeline

The pipeline below validates LLM-extracted ticket fields through three layers: schema, semantic, and behavioural (golden set). All three functions contain a bug that causes silent validation failures — outputs that should fail pass through undetected.

**Your task:** Identify each bug and write a fix. The test harness at the end exposes all three failures.

## Setup

In [ ]:
import json
from datetime import date, timedelta
from pathlib import Path
from pydantic import BaseModel, ValidationError, validator

# Note: this imports 'validator' — think about which version of Pydantic is installed
print("Pydantic version:", __import__('pydantic').VERSION)

## Layer 1: Schema validation

The `SupportTicket` model should enforce that `priority` is between 1 and 5.

In [ ]:
class SupportTicket(BaseModel):
    customer_name: str
    issue_category: str
    priority: int
    requires_escalation: bool

    @validator('priority')
    @classmethod
    def priority_in_range(cls, v: int) -> int:
        if not 1 <= v <= 5:
            raise ValueError(f'priority must be 1-5, got {v}')
        return v

## Layer 2: Semantic validation

Escalation rule: tickets with `priority >= 4` **and** `requires_escalation == False` are inconsistent — high-priority tickets must be escalated.

In [ ]:
def validate_ticket_semantics(ticket: SupportTicket) -> None:
    """
    Raise ValueError if the ticket fails semantic consistency rules.
    """
    # Rule: high-priority tickets must be escalated
    if ticket.priority >= 4 or not ticket.requires_escalation:
        raise ValueError(
            f'High-priority ticket (priority={ticket.priority}) '
            'must have requires_escalation=True'
        )

    # Rule: customer_name must not be a placeholder
    placeholders = {'n/a', 'unknown', 'null', 'none', 'customer'}
    if ticket.customer_name.lower().strip() in placeholders:
        raise ValueError(
            f'customer_name looks like a placeholder: {ticket.customer_name!r}'
        )

## Layer 3: Behavioural validation (golden set)

In [ ]:
GOLDEN_SET = [
    {"input": "Urgent: payment system down, all transactions failing",
     "expected": {"issue_category": "technical", "priority": 5, "requires_escalation": True}},
    {"input": "My invoice shows wrong amount, billed $450 instead of $400",
     "expected": {"issue_category": "billing", "priority": 2, "requires_escalation": False}},
    {"input": "Package arrived damaged, need replacement",
     "expected": {"issue_category": "shipping", "priority": 3, "requires_escalation": False}},
]


def run_golden_set_check(
    pipeline_fn,
    golden_set: list,
    tolerance: float = 0.95,
) -> dict:
    """
    Run pipeline_fn over the golden set and report accuracy.
    Raises AssertionError if accuracy < tolerance.
    """
    results = {"total": 0, "passed": 0, "failed": [], "errors": []}

    for item in golden_set:
        results["total"] += 1
        try:
            output = pipeline_fn(item["input"])
            expected = item["expected"]
            if (
                output.issue_category == expected["issue_category"]
                and output.priority == expected["priority"]
            ):
                results["passed"] += 1
            else:
                results["failed"].append(
                    {"input": item["input"][:80], "expected": expected,
                     "got": output.model_dump()}
                )
        except Exception as e:
            results["errors"].append({"input": item["input"][:80], "error": str(e)})

    accuracy = results["passed"] / results["total"] if results["total"] else 0.0
    assert accuracy >= tolerance, (
        f"Golden set accuracy {accuracy:.1%} below tolerance {tolerance:.1%}\n"
        f"Failed: {results['failed']}\nErrors: {results['errors']}"
    )
    return results

## Test harness

Run the cells below to see each bug manifest. The output will tell you *what* is wrong; your job is to find *why* and fix it.

In [ ]:
# Test 1: schema validation should reject priority=7
bad_ticket_data = {
    "customer_name": "Alice",
    "issue_category": "billing",
    "priority": 7,
    "requires_escalation": False,
}

try:
    t = SupportTicket(**bad_ticket_data)
    print(f'FAIL: priority=7 was accepted — validator did not run. Got: {t}')
except ValidationError as e:
    print(f'PASS: ValidationError raised as expected: {e.errors()[0]["msg"]}')

In [ ]:
# Test 2: semantic validation should catch priority=5 with requires_escalation=False
high_priority_no_escalation = SupportTicket(
    customer_name="Bob",
    issue_category="technical",
    priority=5,
    requires_escalation=False,
)

try:
    validate_ticket_semantics(high_priority_no_escalation)
    print('FAIL: priority=5 with requires_escalation=False passed semantic validation')
except ValueError as e:
    print(f'PASS: ValueError raised: {e}')

In [ ]:
# Test 3: golden set check must be stable across runs — no state leakage
# A mock pipeline that always returns the first golden set item's expected values
def mock_pipeline(ticket_text: str) -> SupportTicket:
    return SupportTicket(
        customer_name="Test",
        issue_category="technical",
        priority=5,
        requires_escalation=True,
    )

# Run the check twice; results should be identical
r1 = run_golden_set_check(mock_pipeline, GOLDEN_SET, tolerance=0.0)
r2 = run_golden_set_check(mock_pipeline, GOLDEN_SET, tolerance=0.0)

print(f'Run 1 — total: {r1["total"]}, passed: {r1["passed"]}')
print(f'Run 2 — total: {r2["total"]}, passed: {r2["passed"]}')
if r1["total"] == r2["total"]:
    print('State check: totals match (no leakage)')
else:
    print('FAIL: totals differ between runs — state is leaking between calls')

## Find the bugs

Look at the test output above, trace back to the relevant function, and write your diagnosis and fix below.

In [ ]:
# Bug 1 — in SupportTicket / priority_in_range
#
# Diagnosis:
#
#
# Fix (rewrite the model so the validator actually runs on Pydantic v2):
from pydantic import BaseModel, field_validator, ValidationError

class SupportTicketFixed(BaseModel):
    customer_name: str
    issue_category: str
    priority: int
    requires_escalation: bool

    # Add your fixed validator here
    pass

In [ ]:
# Bug 2 — in validate_ticket_semantics
#
# Diagnosis:
#
#
# Fix (rewrite the escalation rule with the correct logical operator):
def validate_ticket_semantics_fixed(ticket) -> None:
    pass  # replace with your implementation

In [ ]:
# Bug 3 — in run_golden_set_check
#
# Diagnosis: (hint — what happens to results['total'] across calls if
#  the golden_set list is shared between callers?)
#
#
# There is no mutable state leak in the current code — re-read the test output.
# The bug is subtler: look at what the function accepts as golden_set
# and think about what happens when the caller passes a module-level list
# that another caller could mutate before the check runs.
# Fix: make run_golden_set_check defensive against a mutated golden_set.
def run_golden_set_check_fixed(pipeline_fn, golden_set: list, tolerance: float = 0.95) -> dict:
    pass  # replace with your implementation